# Load Raw Weather Data from Bronze Layer

This step loads the raw weather data previously stored in the Bronze layer into a Spark DataFrame.

The Bronze table serves as the source for all subsequent data transformations in the Silver layer.

No transformations are applied at this stage.

In [0]:
# Define the source Bronze table
BRONZE_TABLE = "Workspace.weather_data.weather_raw"

# Load raw weather data from Bronze table
bronze_df = spark.read.table(BRONZE_TABLE)

# Display the DataFrame schema
bronze_df.printSchema()

# Display the first 5 records
display(bronze_df.limit(5))

# Count total number of records
print(f"Total nubmber of records: {bronze_df.count()}")


root
 |-- apparent_temperature: double (nullable = true)
 |-- city: string (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- measurement_time: string (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- shortwave_radiation: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- wind_direction_10m: long (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



apparent_temperature,city,cloud_cover,measurement_time,precipitation,rain,relative_humidity_2m,shortwave_radiation,snowfall,temperature_2m,wind_direction_10m,wind_gusts_10m,wind_speed_10m,latitude,longitude
-2.4,Warszawa,75,2020-01-01T00:00,0.0,0.0,76,0.0,0.0,2.8,302,39.6,19.9,52.2297,21.0122
-2.2,Warszawa,51,2020-01-01T01:00,0.0,0.0,77,0.0,0.0,2.8,302,37.1,18.6,52.2297,21.0122
-3.1,Warszawa,0,2020-01-01T02:00,0.0,0.0,83,0.0,0.0,1.5,294,34.2,15.8,52.2297,21.0122
-4.0,Warszawa,0,2020-01-01T03:00,0.0,0.0,87,0.0,0.0,0.7,291,29.5,16.2,52.2297,21.0122
-4.4,Warszawa,0,2020-01-01T04:00,0.0,0.0,90,0.0,0.0,0.1,291,29.5,15.0,52.2297,21.0122


Total nubmber of records: 359136


# Convert Measurement Time to Timestamp

This step converts the measurement_time column from string to timestamp.

Using the timestamp data type enables proper time-based filtering, sorting, and aggregation in subsequent processing stages.

All other columns remain unchanged.

In [0]:
from pyspark.sql.functions import col, to_timestamp

# Convert measurement_time column from string to timestamp
silver_df = bronze_df.withColumn(
    "measurement_time", to_timestamp(col("measurement_time"))
)

# Display the updated DataFrame schema
silver_df.printSchema()

# Display top 5 rows
display(silver_df.limit(5))

root
 |-- apparent_temperature: double (nullable = true)
 |-- city: string (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- measurement_time: timestamp (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- shortwave_radiation: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- wind_direction_10m: long (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



apparent_temperature,city,cloud_cover,measurement_time,precipitation,rain,relative_humidity_2m,shortwave_radiation,snowfall,temperature_2m,wind_direction_10m,wind_gusts_10m,wind_speed_10m,latitude,longitude
-2.4,Warszawa,75,2020-01-01T00:00:00.000Z,0.0,0.0,76,0.0,0.0,2.8,302,39.6,19.9,52.2297,21.0122
-2.2,Warszawa,51,2020-01-01T01:00:00.000Z,0.0,0.0,77,0.0,0.0,2.8,302,37.1,18.6,52.2297,21.0122
-3.1,Warszawa,0,2020-01-01T02:00:00.000Z,0.0,0.0,83,0.0,0.0,1.5,294,34.2,15.8,52.2297,21.0122
-4.0,Warszawa,0,2020-01-01T03:00:00.000Z,0.0,0.0,87,0.0,0.0,0.7,291,29.5,16.2,52.2297,21.0122
-4.4,Warszawa,0,2020-01-01T04:00:00.000Z,0.0,0.0,90,0.0,0.0,0.1,291,29.5,15.0,52.2297,21.0122


# Validate Measurement Time Conversion

This step verifies that the measurement_time column was successfully converted to timestamp.

It checks whether any null values were introduced during the conversion.

In [0]:
# Count records with null measurement_time values
null_timestamps = silver_df.filter(col("measurement_time").isNull()).count()

# Display number of records with null measurement_time
print(f"Number of records with null measurement_time: {null_timestamps}")

Number of records with null measurement_time: 0


# Validate Null Values in City

This step checks whether the city column contains null values.

Every weather record must be associated with a valid city.
The pipeline must stop if null values are detected.

In [0]:
# Count null values in the city column
city_null_values = silver_df.filter(col("city").isNull()).count()

# Stop the pipeline if null values are detected
if city_null_values > 0:
    raise ValueError(
        "Data quality check failed: {city_null_values} detected in the city column"
    )
print("Data quality check passed: No Null values in the city column")

Data quality check passed: No Null values in the city column


# Validate Empty Values in City

This step checks whether the city column contains empty strings or values consisting only of whitespace.

Every weather record must contain a non-empty city name.
The pipeline must stop if empty values are detected.

In [0]:
from pyspark.sql.functions import trim

# Count empty or whitespace-only city values
empty_city_count = silver_df.filter(trim(col("city")) == "").count()

# Stop the pipeline if empty values are detected
if empty_city_count > 0:
    raise ValueError(f"Data quality check failed: {empty_city_count} empty values found in city column")
else:
    print("Data quality check passed: No empty values found in city column")

Data quality check passed: No empty values found in city column


# Check Leading and Trailing Whitespace in City Names

This step identifies city names containing leading or trailing whitespace.

The original string length is compared with the trimmed string length to detect unnecessary whitespace.

In [0]:
from pyspark.sql.functions import length

# Identify city names with leading or trailing whitespaces
whitespace_city_df = (
    silver_df.filter(length(trim(col("city"))) != length(col("city")))
    .select("city")
    .distinct()
)

# Display detected city names
display(whitespace_city_df)

# Count number of cities with leading or trailing whitespaces
whitespace_count = whitespace_city_df.count()

if whitespace_count > 0:
    print(
        f"Data quality check failed: number of distinct values with leading or trailing whitespaces in city column: {whitespace_count}"
    )
else:
    print(
        "Data quality check passed: No Leading or Trailing whitespaces found in city column"
    )

city


Data quality check passed: No Leading or Trailing whitespaces found in city column


# Remove Leading and Trailing Whitespace

This step removes leading and trailing whitespace from city names when necessary.

The transformation is applied only if whitespace issues have been detected.

In [0]:
# # Check whether leading or trailing whitespace was detected
trim_applied = whitespace_count > 0

# Apply trim() only when necessary
if trim_applied:
    silver_df = silver_df.withColumn("city", trim("city"))
    print("Leading or trailing spaces removed from city column")
else:
    print("No leading or trailing spaces found in city column")

No leading or trailing spaces found in city column


# Validate City Names After Whitespace Removal

This step verifies that no leading or trailing whitespace remains in the city column after the transformation.

The pipeline stops if any whitespace issues are still detected.

In [0]:
if trim_applied:
    # Count city values with leading or trailing whitespace
    remaining_whitespace_city_count = silver_df.filter(
        length(trim(col("city"))) != length(col("city"))
    ).count()
    if remaining_whitespace_city_count == 0:
        print("Data quality check passed: ")
        print("No leading or trailing spaces found in city column")
    else:
        raise ValueError(
            f"Data quality check failed: {remaining_whitespace_city_count} records still containing leading or trailing whitespaces in city column"
        )
else:
    print(
        "Data quality check passed: No whitespaces transformation was reqiured"
    )

Data quality check passed: No whitespaces transformation was reqiured


# Validate Uniqueness of City and Measurement Time

This step verifies that each city has only one record for each measurement_time.

The combination of city and measurement_time must be unique.

The pipeline stops if duplicate combinations are detected.

In [0]:
# Count duplicate city and measurement_time combinations
duplicate_count = (
    silver_df.groupBy("city", "measurement_time").count().filter(col("count") > 1).count())
display(duplicate_count)

# Stop the pipeline if duplicate combinations are detected
if duplicate_count > 0:
    raise ValueError(f"Data quality check failed: {duplicate_count} city and measurement_time combinations are duplicated")
else:
    print("Data quality check passed: city and measurement_time combinations are unique")

0

Data quality check passed: city and measurement_time combinations are unique


# Add Time Attributes

This step derives additional time attributes from the measurement_time column.

The new columns will support time-based filtering and aggregations in the Gold layer.

The original measurement_time column remains unchanged.

In [0]:
from pyspark.sql.functions import to_date, year, month, quarter, dayofmonth

# Add time-related columns
silver_df = (
    silver_df.withColumn("date", to_date(col("measurement_time")))
    .withColumn("year", year(col("measurement_time")))
    .withColumn("month", month(col("measurement_time")))
    .withColumn("quarter", quarter(col("measurement_time")))
    .withColumn("day_of_month", dayofmonth(col("measurement_time"))) 
)

# Display the updated DataFrame schema
display(silver_df.schema)

# Display the first 5 rows of the updated DataFrame
display(silver_df.limit(10))

StructType([StructField('apparent_temperature', DoubleType(), True), StructField('city', StringType(), True), StructField('cloud_cover', LongType(), True), StructField('measurement_time', TimestampType(), True), StructField('precipitation', DoubleType(), True), StructField('rain', DoubleType(), True), StructField('relative_humidity_2m', LongType(), True), StructField('shortwave_radiation', DoubleType(), True), StructField('snowfall', DoubleType(), True), StructField('temperature_2m', DoubleType(), True), StructField('wind_direction_10m', LongType(), True), StructField('wind_gusts_10m', DoubleType(), True), StructField('wind_speed_10m', DoubleType(), True), StructField('date', DateType(), True), StructField('year', IntegerType(), True), StructField('month', IntegerType(), True), StructField('quarter', IntegerType(), True), StructField('day_of_month', IntegerType(), True)])

apparent_temperature,city,cloud_cover,measurement_time,precipitation,rain,relative_humidity_2m,shortwave_radiation,snowfall,temperature_2m,wind_direction_10m,wind_gusts_10m,wind_speed_10m,date,year,month,quarter,day_of_month
19.9,Mińsk Mazowiecki,73,2021-09-15T12:00:00.000Z,0.0,0.0,57,423.0,0.0,20.6,208,20.9,8.5,2021-09-15,2021,9,3,15
20.4,Mińsk Mazowiecki,88,2021-09-15T13:00:00.000Z,0.0,0.0,60,428.0,0.0,20.7,174,19.8,7.2,2021-09-15,2021,9,3,15
20.5,Mińsk Mazowiecki,72,2021-09-15T14:00:00.000Z,0.0,0.0,58,406.0,0.0,20.9,169,18.4,7.3,2021-09-15,2021,9,3,15
20.4,Mińsk Mazowiecki,95,2021-09-15T15:00:00.000Z,0.0,0.0,57,300.0,0.0,20.7,153,17.6,6.4,2021-09-15,2021,9,3,15
20.5,Mińsk Mazowiecki,97,2021-09-15T16:00:00.000Z,0.0,0.0,58,191.0,0.0,20.6,125,14.4,4.4,2021-09-15,2021,9,3,15
19.6,Mińsk Mazowiecki,97,2021-09-15T17:00:00.000Z,0.0,0.0,73,97.0,0.0,18.9,81,9.0,4.4,2021-09-15,2021,9,3,15
16.9,Mińsk Mazowiecki,75,2021-09-15T18:00:00.000Z,0.0,0.0,73,17.0,0.0,17.2,99,11.2,6.9,2021-09-15,2021,9,3,15
14.5,Mińsk Mazowiecki,86,2021-09-15T19:00:00.000Z,0.0,0.0,79,0.0,0.0,15.1,101,11.9,7.3,2021-09-15,2021,9,3,15
14.3,Mińsk Mazowiecki,87,2021-09-15T20:00:00.000Z,0.0,0.0,75,0.0,0.0,15.3,99,14.4,8.8,2021-09-15,2021,9,3,15
13.5,Mińsk Mazowiecki,81,2021-09-15T21:00:00.000Z,0.0,0.0,78,0.0,0.0,14.6,105,16.2,9.7,2021-09-15,2021,9,3,15


# Validate Time Attributes

This step verifies that all derived time attributes contain valid values.

The pipeline stops if any of the new time-related columns contain NULL values.

In [0]:
# Define the derived time columns to validate
time_columns = ["date", "year", "month", "quarter", "day_of_month"]

# Count NULL values in each derived time column
for column in time_columns:
    null_count = silver_df.filter(col(column).isNull()).count()

    if null_count > 0:
        raise ValueError(f"Data quality check failed: {null_count} Null values found in {column}")
    else:
        print(f"Data quality check passed: No Null values found in {column}")

Data quality check passed: No Null values found in date
Data quality check passed: No Null values found in year
Data quality check passed: No Null values found in month
Data quality check passed: No Null values found in quarter
Data quality check passed: No Null values found in day_of_month


# Reorder Silver DataFrame Columns

This step rearranges the columns in the Silver DataFrame to improve readability and logical organization.

Columns are grouped by their meaning, with city and measurement_time placed at the beginning.

No data values are modified or removed.

In [0]:
# Define the desired column order
column_order = [
    "city",
    "measurement_time",
    "date",
    "year",
    "month",
    "quarter",
    "day_of_month",
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "cloud_cover",
    "wind_speed_10m",
    "wind_gusts_10m",
    "wind_direction_10m",
    "shortwave_radiation"
]   

# Reorder DataFrame columns
silver_df = silver_df.select(*column_order)

# Display the updated DataFrame
display(silver_df.limit(10))

city,measurement_time,date,year,month,quarter,day_of_month,temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,rain,snowfall,cloud_cover,wind_speed_10m,wind_gusts_10m,wind_direction_10m,shortwave_radiation
Mińsk Mazowiecki,2021-09-15T12:00:00.000Z,2021-09-15,2021,9,3,15,20.6,19.9,57,0.0,0.0,0.0,73,8.5,20.9,208,423.0
Mińsk Mazowiecki,2021-09-15T13:00:00.000Z,2021-09-15,2021,9,3,15,20.7,20.4,60,0.0,0.0,0.0,88,7.2,19.8,174,428.0
Mińsk Mazowiecki,2021-09-15T14:00:00.000Z,2021-09-15,2021,9,3,15,20.9,20.5,58,0.0,0.0,0.0,72,7.3,18.4,169,406.0
Mińsk Mazowiecki,2021-09-15T15:00:00.000Z,2021-09-15,2021,9,3,15,20.7,20.4,57,0.0,0.0,0.0,95,6.4,17.6,153,300.0
Mińsk Mazowiecki,2021-09-15T16:00:00.000Z,2021-09-15,2021,9,3,15,20.6,20.5,58,0.0,0.0,0.0,97,4.4,14.4,125,191.0
Mińsk Mazowiecki,2021-09-15T17:00:00.000Z,2021-09-15,2021,9,3,15,18.9,19.6,73,0.0,0.0,0.0,97,4.4,9.0,81,97.0
Mińsk Mazowiecki,2021-09-15T18:00:00.000Z,2021-09-15,2021,9,3,15,17.2,16.9,73,0.0,0.0,0.0,75,6.9,11.2,99,17.0
Mińsk Mazowiecki,2021-09-15T19:00:00.000Z,2021-09-15,2021,9,3,15,15.1,14.5,79,0.0,0.0,0.0,86,7.3,11.9,101,0.0
Mińsk Mazowiecki,2021-09-15T20:00:00.000Z,2021-09-15,2021,9,3,15,15.3,14.3,75,0.0,0.0,0.0,87,8.8,14.4,99,0.0
Mińsk Mazowiecki,2021-09-15T21:00:00.000Z,2021-09-15,2021,9,3,15,14.6,13.5,78,0.0,0.0,0.0,81,9.7,16.2,105,0.0


# Save Transformed Data to Silver Layer

This step saves the cleaned, validated, and organized weather data to a managed Delta table in the Silver layer.

The Silver table serves as the source for subsequent aggregations in the Gold layer.

In [0]:
# Define the Silver layer table name
SILVER_TABLE = "Workspace.weather_data.weather_silver"

# Save the transformed data as a Delta table
silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

# Confirm successful data storage
print(f"Data successfully stored in the {SILVER_TABLE}")

Data successfully stored in the Workspace.weather_data.weather_silver


# Validate Silver Table

This step verifies that the transformed weather data has been successfully saved to the Silver layer.

The validation checks whether the table can be loaded and whether its column names and order match the structure defined during the transformation stage.


In [0]:
# Load the saved Silver table
silver_saved_df = spark.read.table(SILVER_TABLE)

# Validate the column names and their order
if silver_saved_df.columns != column_order:
      raise ValueError(
        "Silver table validation failed: column names or their order do not match the expected structure."
    )
else:
    print("Silver table validation passed: column names and their order match the expected structure.")

# Report total number of records
print(f"Total number of records in the Silver table: {silver_saved_df.count()}")

# Report total number of columns
print(f"Total number of columns in the Silver table: {len(silver_saved_df.columns)}")

# Confirm successful validation
print("Silver table validation completed successfully.")


Silver table validation passed: column names and their order match the expected structure.
Total number of records in the Silver table: 359136
Total number of columns in the Silver table: 18
Silver table validation completed successfully.


In [0]:
spark.conf.get("spark.sql.session.timeZone")

'Etc/UTC'